In [1]:
import re

import numpy as np
import pandas as pd
from loguru import logger
from scipy.stats import kendalltau, spearmanr
from sklearn.metrics import average_precision_score, cohen_kappa_score

## Define Metrics & Utils

In [2]:
def parse_annotation_text(text):
    """Parse [HAL]...[/HAL] annotation markup into span tuples and binary labels."""
    if pd.isna(text) or text == "":
        return [], []

    hal_pattern = r"\[HAL\](.*?)\[/HAL\]"
    hal_spans = list(re.finditer(hal_pattern, text))
    clear_text = re.sub(r"\[/?HAL\]", "", text)

    bias = 0
    spans = []
    labels = np.zeros(len(clear_text), dtype=int)
    for match in hal_spans:
        start_idx = match.start() - bias
        bias += len("[HAL][/HAL]")
        end_idx = match.end() - bias
        labels[start_idx:end_idx] = 1
        spans.append((start_idx, end_idx))

    return spans, labels.tolist()


def find_spans_of_ones(arr):
    """Return contiguous (start, end) pairs for positive regions."""
    arr = np.ceil(arr)
    padded = np.pad(arr, pad_width=1, mode="constant", constant_values=0)
    diff = np.diff(padded)
    starts = np.where(diff == 1)[0]
    ends = np.where(diff == -1)[0]
    return list(zip(starts, ends))


def merge_annotation_labels(*labels, merge_type="union"):
    """Merge multiple annotation label arrays according to the requested strategy."""
    if merge_type == "union":
        merged = np.maximum.reduce(labels)
    elif merge_type == "hard-intersection":
        merged = np.minimum.reduce(labels)
    elif merge_type == "soft-intersection":
        merged = np.mean(labels, axis=0)
    else:
        raise ValueError(f"Unsupported merge_type: {merge_type}")

    return merged.tolist()


def parse_and_merge_annotations(*texts, merge_type="union"):
    """Parse each annotation text and merge their label arrays."""
    if len(texts) < 2:
        return parse_annotation_text(texts[0])

    parsed = [parse_annotation_text(text) for text in texts]
    labels_list = [labels for tokens, labels in parsed]

    merged_labels = merge_annotation_labels(*labels_list, merge_type=merge_type)
    merged_tokens = find_spans_of_ones(merged_labels)

    return merged_tokens, merged_labels


def calculate_basic_metrics(gpt_text, expert_texts, merge_type="union"):
    """Compute core annotation overlap metrics between GPT output and experts."""
    gpt_tokens, gpt_labels = parse_annotation_text(gpt_text)
    expert_tokens, expert_labels = parse_and_merge_annotations(
        *expert_texts, merge_type=merge_type
    )

    try:
        if not gpt_labels and not expert_labels:
            logger.debug("No labels returned for GPT/expert texts; emitting NaNs.")
            return create_nan_metrics(expert_tokens, merge_type)

        gpt_arr = np.asarray(gpt_labels, dtype=int)
        expert_arr = np.asarray(expert_labels, dtype=int)

        if gpt_arr.size != expert_arr.size:
            max_len = max(gpt_arr.size, expert_arr.size)
            gpt_arr = np.pad(gpt_arr, (0, max_len - gpt_arr.size))
            expert_arr = np.pad(expert_arr, (0, max_len - expert_arr.size))

        expert_positive = expert_arr > 0
        gpt_positive = gpt_arr > 0

        tp = np.sum(expert_positive & gpt_positive)
        fp = np.sum(~expert_positive & gpt_positive)
        fn = np.sum(expert_positive & ~gpt_positive)
        tn = np.sum(~expert_positive & ~gpt_positive)

        has_expert_positives = bool(expert_positive.any())
        has_gpt_positives = bool(gpt_positive.any())
        has_any_positives = has_expert_positives or has_gpt_positives

        accuracy = np.mean(expert_arr == gpt_arr)

        if not has_any_positives:
            iou = 1.0
        else:
            denominator = np.sum(expert_positive | gpt_positive)
            iou = (
                np.sum(expert_positive & gpt_positive) / denominator
                if denominator
                else 0.0
            )

        label_union = np.unique(np.concatenate((expert_arr, gpt_arr)))
        can_compute_kappa = has_any_positives and label_union.size > 1

        if can_compute_kappa:
            try:
                kappa = cohen_kappa_score(expert_arr, gpt_arr, labels=[0, 1])
            except Exception as exc:
                logger.warning("Cohen kappa failed: {}", exc)
                kappa = np.nan
        else:
            kappa = 1.0 if np.array_equal(expert_arr, gpt_arr) else np.nan
            if label_union.size <= 1:
                logger.debug(
                    "Skipping Cohen kappa due to single-label inputs (labels={})",
                    label_union.tolist(),
                )

        if np.var(expert_arr) > 0 and np.var(gpt_arr) > 0:
            try:
                kendall_tau, kendall_p = kendalltau(expert_arr, gpt_arr)
                spearman_rho, spearman_p = spearmanr(expert_arr, gpt_arr)
            except Exception as exc:
                logger.warning("Rank correlation failed: {}", exc)
                kendall_tau = kendall_p = spearman_rho = spearman_p = np.nan
        else:
            if np.array_equal(np.ceil(expert_arr), gpt_arr):
                kendall_tau = spearman_rho = 1.0
                kendall_p = spearman_p = 0.0
            else:
                kendall_tau = kendall_p = spearman_rho = spearman_p = np.nan

        precision = tp / (tp + fp) if (tp + fp) > 0 else float(not has_expert_positives)
        recall = tp / (tp + fn) if (tp + fn) > 0 else float(not has_gpt_positives)
        f1 = (
            (2 * precision * recall / (precision + recall))
            if (precision + recall)
            else 0.0
        )

        if has_expert_positives:
            try:
                average_prec = average_precision_score(gpt_arr, expert_arr)
            except Exception as exc:
                logger.warning("Average precision failed: {}", exc)
                average_prec = np.nan
        else:
            average_prec = 0.0 if has_gpt_positives else 1.0

        return {
            "cohen_kappa": kappa,
            "kendall_tau": kendall_tau,
            "kendall_p_value": kendall_p,
            "spearman_rho": spearman_rho,
            "spearman_p_value": spearman_p,
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
            "average_precision": average_prec,
            "iou": iou,
            "chars_count": int(expert_arr.size),
            "tp": int(tp),
            "fp": int(fp),
            "fn": int(fn),
            "tn": int(tn),
            "has_expert_positives": has_expert_positives,
            "has_gpt_positives": has_gpt_positives,
            f"{merge_type}_merged_spans": expert_tokens,
        }

    except Exception as exc:
        logger.exception("calculate_basic_metrics failed: {}", exc)
        return create_nan_metrics(expert_tokens, merge_type)


def calculate_fleiss_kappa(annotations):
    """
    Calculate Fleiss' kappa for inter-annotator agreement.

    Args:
        annotations: List of lists, where each inner list contains binary labels
                    from one annotator. Shape: (n_annotators, n_items)

    Returns:
        float: Fleiss' kappa score
    """
    annotations = np.array(annotations)
    n_raters = annotations.shape[0]
    n_items = annotations.shape[1]
    n_categories = len(np.unique(annotations))

    agreement_table = np.zeros((n_items, n_categories), dtype=int)
    for item_idx in range(n_items):
        for category in range(n_categories):
            agreement_table[item_idx, category] = np.sum(
                annotations[:, item_idx] == category
            )

    P_i = np.zeros(n_items)
    for i in range(n_items):
        P_i[i] = (np.sum(agreement_table[i, :] ** 2) - n_raters) / (
            n_raters * (n_raters - 1)
        )

    P_bar = np.mean(P_i)

    p_j = np.sum(agreement_table, axis=0) / (n_items * n_raters)
    P_e = np.sum(p_j**2)

    if P_e == 1:
        return 1.0

    kappa = (P_bar - P_e) / (1 - P_e)
    return kappa


def calculate_interannotator_agreement(expert_texts, method="fleiss"):
    """
    Calculate inter-annotator agreement between multiple expert annotations.

    Args:
        expert_texts: List of expert annotation texts with [HAL] markup
        method: "fleiss" for Fleiss' kappa or "pairwise_cohen" for average pairwise Cohen's kappa

    Returns:
        dict: Agreement metrics including Fleiss' kappa and other statistics
    """
    if len(expert_texts) < 2:
        raise ValueError("Need at least 2 annotators for inter-annotator agreement")

    parsed_annotations = [parse_annotation_text(text) for text in expert_texts]
    labels_list = [labels for _, labels in parsed_annotations]

    max_len = max(len(labels) for labels in labels_list)
    aligned_labels = []
    for labels in labels_list:
        padded = labels + [0] * (max_len - len(labels))
        aligned_labels.append(padded[:max_len])

    aligned_labels = np.array(aligned_labels)
    logger.debug(
        "Computing inter-annotator agreement for {} annotators over {} tokens",
        aligned_labels.shape[0],
        aligned_labels.shape[1],
    )

    results = {}
    if method in {"fleiss", "both"}:
        fleiss_k = calculate_fleiss_kappa(aligned_labels)
        results["fleiss_kappa"] = fleiss_k

        n_items = aligned_labels.shape[1]
        n_annotators = aligned_labels.shape[0]
        full_agreement = (
            np.sum(np.all(aligned_labels == aligned_labels[0, :], axis=0)) / n_items
        )
        results["full_agreement_rate"] = full_agreement

        majority_agreement = np.sum(np.mean(aligned_labels, axis=0) >= 0.5)
        results["majority_positive_items"] = majority_agreement

        pairwise_agreements = []
        for i in range(n_annotators):
            for j in range(i + 1, n_annotators):
                agreement = np.mean(aligned_labels[i] == aligned_labels[j])
                pairwise_agreements.append(agreement)
        results["avg_pairwise_agreement"] = np.mean(pairwise_agreements)

    if method in {"pairwise_cohen", "both"}:
        from itertools import combinations

        pairwise_kappas = []
        skipped_pairs = 0
        for i, j in combinations(range(len(aligned_labels)), 2):
            combined_labels = np.unique(
                np.concatenate((aligned_labels[i], aligned_labels[j]))
            )
            if combined_labels.size < 2:
                skipped_pairs += 1
                pairwise_kappas.append(np.nan)
                continue

            try:
                kappa = cohen_kappa_score(aligned_labels[i], aligned_labels[j])
                pairwise_kappas.append(kappa)
            except Exception as exc:
                logger.warning(
                    "Pairwise Cohen kappa failed for annotators ({}, {}): {}",
                    i,
                    j,
                    exc,
                )
                pairwise_kappas.append(np.nan)

        if skipped_pairs:
            logger.info(
                "Skipped {} pairwise kappa computations due to single-label pairs",
                skipped_pairs,
            )

        finite_kappas = np.array([k for k in pairwise_kappas if np.isfinite(k)])
        results["pairwise_cohen_kappas"] = pairwise_kappas
        results["avg_pairwise_cohen_kappa"] = (
            float(finite_kappas.mean()) if finite_kappas.size else np.nan
        )
        results["std_pairwise_cohen_kappa"] = (
            float(finite_kappas.std(ddof=0)) if finite_kappas.size else np.nan
        )

    results["n_annotators"] = len(aligned_labels)
    results["n_items"] = max_len
    results["annotator_positive_rates"] = [np.mean(labels) for labels in aligned_labels]

    return results


def calculate_metrics(gpt_text, expert_texts, merge_type="union", calculate_iaa=True):
    """
    Extended version that also calculates inter-annotator agreement if requested.
    """
    logger.debug(
        "Calculating metrics with {} expert annotations (merge={}, iaa={})",
        len(expert_texts),
        merge_type,
        calculate_iaa,
    )
    metrics = calculate_basic_metrics(gpt_text, expert_texts, merge_type)

    if calculate_iaa and len(expert_texts) >= 2:
        iaa_metrics = calculate_interannotator_agreement(expert_texts, method="both")
        metrics.update({f"iaa_{key}": value for key, value in iaa_metrics.items()})
    elif calculate_iaa:
        logger.info(
            "Skipping IAA computation: need at least 2 experts, got {}",
            len(expert_texts),
        )

    return metrics


def create_nan_metrics(expert_tokens, merge_type):
    """Create metrics dictionary with NaN values for invalid cases."""
    return {
        "cohen_kappa": np.nan,
        "kendall_tau": np.nan,
        "kendall_p_value": np.nan,
        "spearman_rho": np.nan,
        "spearman_p_value": np.nan,
        "accuracy": np.nan,
        "precision": np.nan,
        "recall": np.nan,
        "f1_score": np.nan,
        "average_precision": np.nan,
        "iou": np.nan,
        "chars_count": 0,
        "tp": 0,
        "fp": 0,
        "fn": 0,
        "tn": 0,
        "has_expert_positives": False,
        "has_gpt_positives": False,
        f"{merge_type}_merged_spans": expert_tokens,
    }


In [3]:
def process_hallucination_dataset(
    df,
    gpt_column="gpt_markup",
    expert_columns=["expert_markup"],
    sample_id_column=None,
    output_path=None,
    merge_type="union",
    stack_samples=False,
    calculate_iaa=True,
):
    """
    Process a DataFrame of hallucination annotations and calculate per-sample metrics.

    Args:
        df: Source DataFrame containing GPT output and expert annotations.
        gpt_column: Column name containing GPT annotations (with [HAL] markup).
        expert_columns: List of column names containing expert annotations.
        sample_id_column: Optional column used to preserve original identifiers.
        output_path: Optional CSV path for saving the per-sample results.
        merge_type: Strategy used when merging expert annotation labels.
        stack_samples: Concatenate all texts into one before processing if True.
        calculate_iaa: Whether to compute inter-annotator agreement metrics.

    Returns:
        DataFrame with calculated metrics for each sample (or stacked corpus).
    """
    required_cols = [gpt_column] + expert_columns
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        logger.error("Missing required columns: {}", missing_cols)
        raise ValueError(f"Missing columns: {missing_cols}")

    if sample_id_column and sample_id_column not in df.columns:
        logger.warning(
            "sample_id_column '{}' not found; defaulting to DataFrame index",
            sample_id_column,
        )
        sample_id_column = None

    total_samples = len(df)
    logger.info(
        "Processing {} samples (merge={}, stack={}, iaa={})",
        total_samples,
        merge_type,
        stack_samples,
        calculate_iaa,
    )

    metrics_list = []
    if stack_samples:
        logger.info("Stacking {} samples before evaluation", total_samples)
        gpt_concat = "".join(str(t) for t in df[gpt_column].dropna())
        expert_concat = [
            "".join(str(t) for t in df[col].dropna()) for col in expert_columns
        ]

        metrics = calculate_metrics(
            gpt_concat,
            expert_concat,
            merge_type=merge_type,
            calculate_iaa=calculate_iaa,
        )

        metrics["sample_id"] = "stacked"
        metrics_list.append(metrics)

        results_df = pd.DataFrame(metrics_list)

    else:
        subset_cols = [gpt_column] + expert_columns
        if sample_id_column and sample_id_column not in subset_cols:
            subset_cols.append(sample_id_column)

        data_iter = df[subset_cols].itertuples(index=False, name=None)
        for processed, (row_index, row_values) in enumerate(
            zip(df.index, data_iter), start=1
        ):
            gpt_text = row_values[0]
            expert_values = row_values[1 : 1 + len(expert_columns)]
            metrics = calculate_metrics(
                gpt_text,
                list(expert_values),
                merge_type=merge_type,
                calculate_iaa=calculate_iaa,
            )

            sample_identifier = row_values[-1] if sample_id_column else row_index
            metrics["sample_id"] = sample_identifier
            metrics_list.append(metrics)

            if processed % 100 == 0 or processed == total_samples:
                logger.info("Processed {}/{} samples", processed, total_samples)

        results_df = pd.DataFrame(metrics_list)

        if sample_id_column:
            results_df = results_df.merge(
                df, left_on="sample_id", right_on=sample_id_column, how="left"
            )
        else:
            results_df = pd.concat([df.reset_index(drop=True), results_df], axis=1)

    if output_path:
        results_df.to_csv(output_path, index=False)
        logger.info("Results saved to {}", output_path)

    return results_df


def calculate_dataset_statistics(results_df):
    """Calculate summary statistics across all samples."""
    metrics_cols = [
        "cohen_kappa",
        "kendall_tau",
        "spearman_rho",
        "accuracy",
        "precision",
        "recall",
        "f1_score",
        "average_precision",
        "iou",
        "iaa_fleiss_kappa",
        "iaa_avg_pairwise_agreement",
        "chars_count",
    ]

    stats = {}
    for metric in metrics_cols:
        if metric in results_df.columns:
            values = results_df[metric].dropna()
            stats[metric] = {
                "mean": values.mean(),
                "std": values.std(),
                "median": values.median(),
                "min": values.min(),
                "max": values.max(),
                "count": len(values),
            }

    summary = pd.DataFrame(stats).T
    logger.info("Calculated dataset statistics for {} metrics", len(summary.index))
    return summary

## Test Consistency

In [4]:
def normalize_annotation_text(text):
    """Collapse whitespace artifacts in annotation strings."""
    return text.replace("\n", " ").replace("\t", " ").strip()


df = pd.read_csv("gpt.csv")
df["gpt"] = df["annotated_span"].apply(normalize_annotation_text)

for i in range(3):
    annotation = pd.read_csv(f"{i + 1}_annotator.csv")
    df[f"{i + 1}_annotator"] = annotation["annotated_span"].apply(
        normalize_annotation_text
    )

In [5]:
results = process_hallucination_dataset(
    df,
    gpt_column="gpt",
    expert_columns=[f"{i + 1}_annotator" for i in range(3)],
    output_path="results.csv",
    merge_type="union",
)
stats = calculate_dataset_statistics(results)

2025-11-18 20:30:50.299 | INFO     | __main__:process_hallucination_dataset:41 - Processing 50 samples (merge=union, stack=False, iaa=True)
2025-11-18 20:30:50.302 | DEBUG    | __main__:calculate_metrics:325 - Calculating metrics with 3 expert annotations (merge=union, iaa=True)
2025-11-18 20:30:50.303 | DEBUG    | __main__:calculate_basic_metrics:117 - Skipping Cohen kappa due to single-label inputs (labels=[0])
2025-11-18 20:30:50.303 | DEBUG    | __main__:calculate_interannotator_agreement:245 - Computing inter-annotator agreement for 3 annotators over 19 tokens
2025-11-18 20:30:50.304 | INFO     | __main__:calculate_interannotator_agreement:300 - Skipped 3 pairwise kappa computations due to single-label pairs
2025-11-18 20:30:50.305 | DEBUG    | __main__:calculate_metrics:325 - Calculating metrics with 3 expert annotations (merge=union, iaa=True)
2025-11-18 20:30:50.308 | DEBUG    | __main__:calculate_interannotator_agreement:245 - Computing inter-annotator agreement for 3 annotato